# Chapter 2 — Tools and `tool_use` · Practice

**Goal:** build every piece of a client-side tool agent by hand (tool definitions, `tool_choice`, strict schemas, structured outputs, validation, the `stop_reason` loop), add a harness that also handles Anthropic-schema and server tools, and finish with a small customer-support agent.

- Theory: [README.md](README.md) (read section 2.x before running the matching section here)
- Estimated time: 2–3 hours
- Estimated API cost: **< $0.10** with the default model (`claude-sonnet-5-5`, all live cells once). Every section also has **offline** cells that use a scripted fake API, so you can learn the mechanics with no key and no credit.
- Prerequisites: [00-prerequisites](../../00-prerequisites/README.md) (venv from `requirements.txt`, `.env` with `ANTHROPIC_API_KEY`), [Chapter 1](../01-claude-api-fundamentals/README.md) (Messages API basics, `stop_reason`)

How to read the cells:

| Marker | Meaning |
|---|---|
| **(offline)** | Runs without a key. Uses `ScriptedClaude`, a real `anthropic.Anthropic` client whose HTTP layer replays canned responses and records what you sent. |
| **(live)** | Calls the real API and costs a fraction of a cent. Skip it if you have no key; nothing offline depends on it. |

In [ ]:
import json
import os
import re
import time
from itertools import count
from typing import Annotated, Any, Callable, Literal, Optional

import anthropic
import httpx2
from dotenv import find_dotenv, load_dotenv
from pydantic import BaseModel, BeforeValidator, ValidationError

load_dotenv(find_dotenv())

MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
# Claude Sonnet 5.5 / Opus 5.5 / Fable 5.1 reject forced tool_choice ("any"/"tool") with a 400.
# For the tool_choice comparison we use a model that still accepts it.
# Haiku 4.5 retires "not sooner than October 15, 2026" (model deprecations page). If it is gone,
# set CLAUDE_FORCED_TOOL_MODEL=claude-opus-5 in .env (also accepts forced tool use).
FORCED_TOOL_MODEL = os.getenv("CLAUDE_FORCED_TOOL_MODEL", "claude-haiku-4-5")

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY; only the (live) cells use it
print("anthropic", anthropic.__version__, "| MODEL =", MODEL, "| FORCED_TOOL_MODEL =", FORCED_TOOL_MODEL)

In [ ]:
# Live cells need a key. If this fails you can still run every (offline) cell below.
assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env (see 00-prerequisites)"

### Offline toolkit (run this once)

`ScriptedClaude` gives you a **real** SDK client (`.client`) whose transport answers with the responses you script, in order, and stores every request body in `.requests`. That means the SDK's own parsing, types and `messages.parse()` logic run for real, only the model is fake. The helpers `text()`, `tool_call()` and `reply()` build response JSON in the exact wire format of the Messages API.

In [ ]:
_ids = count(1)


def text(t: str) -> dict:
    return {"type": "text", "text": t}


def tool_call(name: str, tool_input: dict, id: str | None = None) -> dict:
    return {"type": "tool_use", "id": id or f"toolu_{next(_ids):04d}", "name": name, "input": tool_input}


def reply(*blocks: dict, stop_reason: str | None = None) -> dict:
    """A Messages API response body. stop_reason defaults to tool_use/end_turn based on the blocks."""
    if stop_reason is None:
        stop_reason = "tool_use" if any(b["type"] == "tool_use" for b in blocks) else "end_turn"
    return {
        "id": f"msg_{next(_ids):04d}", "type": "message", "role": "assistant", "model": "scripted",
        "content": list(blocks), "stop_reason": stop_reason, "stop_sequence": None,
        "usage": {"input_tokens": 0, "output_tokens": 0},
    }


def api_error(status: int, message: str) -> dict:
    """A scripted HTTP error (e.g. 400 invalid_request_error)."""
    return {"_status": status, "type": "error", "error": {"type": "invalid_request_error", "message": message}}


class ScriptedClaude:
    """Offline stand-in for the Messages API: replays scripted replies, records every request."""

    def __init__(self, *replies: dict):
        self.replies = list(replies)
        self.requests: list[dict] = []
        transport = httpx2.MockTransport(self._handle)
        self.client = anthropic.Anthropic(
            api_key="offline", max_retries=0, http_client=anthropic.DefaultHttpxClient(transport=transport)
        )

    def _handle(self, request: httpx2.Request) -> httpx2.Response:
        self.requests.append(json.loads(request.content))
        if not self.replies:
            return httpx2.Response(500, json={"type": "error", "error": {"type": "api_error", "message": "script exhausted"}})
        body = dict(self.replies.pop(0))
        return httpx2.Response(body.pop("_status", 200), json=body)


def show(response) -> None:
    """Print only what matters: stop_reason, content blocks (thinking collapsed), usage."""
    print("stop_reason:", response.stop_reason)
    for b in response.content:
        if b.type == "text":
            print(f"  [text] {b.text[:300]}")
        elif b.type == "tool_use":
            print(f"  [tool_use] {b.name}({json.dumps(b.input)})  id={b.id}")
        else:
            print(f"  [{b.type}]")
    u = response.usage
    print(f"usage: in={u.input_tokens} out={u.output_tokens}")


print("offline toolkit ready")

### The running example: a tiny support backend

Every section uses the same in-memory "backend" and the same three tools from the README: `get_customer`, `lookup_order`, `process_refund`. Amounts are integer **cents** (no float money).

In [ ]:
CUSTOMERS = {
    48213: {"customer_id": 48213, "name": "Jane Doe", "email": "jane@example.com", "status": "active",
            "order_ids": ["ORD-5001", "ORD-5002"]},
    51877: {"customer_id": 51877, "name": "Omar Haddad", "email": "omar@example.com", "status": "suspended",
            "order_ids": ["ORD-5003"]},
}
ORDERS = {
    "ORD-5001": {"order_id": "ORD-5001", "customer_id": 48213, "status": "delivered", "total_cents": 4999,
                 "refunded_cents": 0, "items": [{"sku": "MUG-01", "name": "Ceramic mug", "qty": 1}]},
    "ORD-5002": {"order_id": "ORD-5002", "customer_id": 48213, "status": "shipped", "total_cents": 129900,
                 "refunded_cents": 0, "items": [{"sku": "LAP-14", "name": "14-inch laptop", "qty": 1}]},
    "ORD-5003": {"order_id": "ORD-5003", "customer_id": 51877, "status": "delivered", "total_cents": 2500,
                 "refunded_cents": 0, "items": [{"sku": "TEE-M", "name": "T-shirt (M)", "qty": 1}]},
}

RICH_TOOLS = [
    {
        "name": "get_customer",
        "description": (
            "Look up a single customer's profile by email address or numeric customer ID. "
            "Returns name, email, account status and a list of order IDs (not order details). "
            "Use this FIRST to verify who the customer is before calling lookup_order or process_refund. "
            "Do NOT use it to answer questions about an order's status or contents; use lookup_order for that. "
            "Provide either email (format user@domain.com) or customer_id, not both."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "email": {"type": "string", "description": "Customer email, e.g. jane@example.com"},
                "customer_id": {"type": "integer", "description": "Numeric customer ID, e.g. 48213"},
            },
            "required": [],
            "additionalProperties": False,
        },
        "input_examples": [{"email": "jane@example.com"}, {"customer_id": 48213}],
    },
    {
        "name": "lookup_order",
        "description": (
            "Fetch one order by its order ID (format ORD-#### e.g. ORD-5001). "
            "Returns status (processing, shipped, delivered), total in cents, amount already refunded and line items. "
            "Use this for any question about an order's status, contents or price. "
            "Do NOT use it to find a customer; it does not search by name or email."
        ),
        "input_schema": {
            "type": "object",
            "properties": {"order_id": {"type": "string", "description": "Order ID, e.g. ORD-5001"}},
            "required": ["order_id"],
            "additionalProperties": False,
        },
    },
    {
        "name": "process_refund",
        "description": (
            "Issue a refund to the original payment method for one order. IRREVERSIBLE: money leaves the account. "
            "Only call after get_customer has verified the customer and lookup_order shows the order belongs to them "
            "and is delivered. amount_cents must not exceed total_cents minus refunded_cents. "
            "Refunds above 50000 cents ($500) are queued for human approval instead of being paid immediately."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "Order ID, e.g. ORD-5001"},
                "amount_cents": {"type": "integer", "description": "Refund amount in cents, e.g. 4999 for $49.99"},
                "reason": {"type": "string", "enum": ["damaged", "not_received", "wrong_item", "other"]},
                "reason_detail": {"type": ["string", "null"],
                                  "description": "Short free-text detail when reason is 'other'; otherwise null."},
            },
            "required": ["order_id", "amount_cents", "reason", "reason_detail"],
            "additionalProperties": False,
        },
    },
]
TOOLS_BY_NAME = {t["name"]: t for t in RICH_TOOLS}
print([t["name"] for t in RICH_TOOLS])

## 2.1 What is `tool_use`

Claude never executes anything. It returns a `tool_use` block (tool name + JSON arguments) with `stop_reason: "tool_use"`; **your code** decides whether to run it, runs it, and sends a `tool_result` back in the next `user` message. Then Claude continues with the result in context. [Theory](README.md#21-what-is-tool_use)

### One round trip by hand (offline)

We script the two responses Claude would send and look at exactly what crosses the wire in each direction.

In [ ]:
fake = ScriptedClaude(
    reply(text("Let me check that order."), tool_call("lookup_order", {"order_id": "ORD-5001"}, id="toolu_A")),
    reply(text("Your order ORD-5001 (a ceramic mug) was delivered.")),
)
messages = [{"role": "user", "content": "Where is my order ORD-5001?"}]

# Request 1: Claude asks for a tool
r1 = fake.client.messages.create(model=MODEL, max_tokens=1024, tools=RICH_TOOLS, messages=messages)
show(r1)

# YOUR code runs the tool (Claude can't)
call = next(b for b in r1.content if b.type == "tool_use")
result = ORDERS[call.input["order_id"]]

# Request 2: full assistant content + a user message holding the tool_result
messages += [
    {"role": "assistant", "content": r1.content},
    {"role": "user", "content": [{"type": "tool_result", "tool_use_id": call.id, "content": json.dumps(result)}]},
]
r2 = fake.client.messages.create(model=MODEL, max_tokens=1024, tools=RICH_TOOLS, messages=messages)
show(r2)

print("\nWhat we actually sent the second time (last two messages):")
print(json.dumps(fake.requests[1]["messages"][-2:], indent=1)[:900])

> **Observe:** the `tool_use_id` in your `tool_result` equals the `id` of Claude's `tool_use` block, and the assistant message you send back contains the **whole** `response.content` (text + `tool_use`), not just the text. The tools array is re-sent on every request: the API is stateless.

> **Try it:** replace `tool_use_id=call.id` with a made-up id such as `"toolu_XYZ"` and rerun. Nothing breaks offline (the fake API doesn't validate), but the real API answers with a 400. That is exactly why section 2.6 builds a validator for these rules.

### The same round trip against the real API (live)

In [ ]:
messages = [{"role": "user", "content": "Where is my order ORD-5001?"}]
r1 = client.messages.create(model=MODEL, max_tokens=1024, tools=RICH_TOOLS, messages=messages)
show(r1)

if r1.stop_reason == "tool_use":
    messages.append({"role": "assistant", "content": r1.content})
    results = [
        {"type": "tool_result", "tool_use_id": b.id, "content": json.dumps(ORDERS.get(b.input.get("order_id"), {}))}
        for b in r1.content if b.type == "tool_use"
    ]
    messages.append({"role": "user", "content": results})
    r2 = client.messages.create(model=MODEL, max_tokens=1024, tools=RICH_TOOLS, messages=messages)
    show(r2)

> **Observe:** on Claude Sonnet 5.5 you may also see a `[thinking]` block. Adaptive thinking is on by default and its text is omitted unless you ask for a summary. Keep it in history anyway (we append the full `content`).

## 2.2 Tool definition

A tool is `name` + `input_schema` (+ optional `description`, `input_examples`, `strict`). The **description is the router**: it is the main signal Claude uses to choose a tool, so state what it does, what it returns, input formats, and when *not* to use it. [Theory](README.md#22-tool-definition)

### Lint your tool definitions (offline)

A small checker for the rules that are mechanical: name pattern, description length/boundaries, and `input_examples` that actually validate against the schema (the API returns a 400 for an invalid example).

In [ ]:
NAME_RE = re.compile(r"^[a-zA-Z0-9_-]{1,128}$")
JSON_TYPES = {"string": str, "integer": int, "number": (int, float), "boolean": bool, "object": dict,
              "array": list, "null": type(None)}


def matches_schema(value: Any, schema: dict) -> list[str]:
    """Tiny JSON Schema subset validator (type, enum, required, properties, additionalProperties, items)."""
    errors: list[str] = []
    types = schema.get("type")
    if types is not None:
        types = types if isinstance(types, list) else [types]
        ok = any(isinstance(value, JSON_TYPES[t]) and not (t in ("integer", "number") and isinstance(value, bool))
                 for t in types)
        if not ok:
            return [f"{value!r} is not of type {types}"]
    if "enum" in schema and value not in schema["enum"]:
        errors.append(f"{value!r} not in enum {schema['enum']}")
    if isinstance(value, dict):
        props = schema.get("properties", {})
        errors += [f"missing required '{k}'" for k in schema.get("required", []) if k not in value]
        if schema.get("additionalProperties") is False:
            errors += [f"unexpected property '{k}'" for k in value if k not in props]
        for k, v in value.items():
            if k in props:
                errors += [f"{k}: {e}" for e in matches_schema(v, props[k])]
    if isinstance(value, list) and "items" in schema:
        for i, v in enumerate(value):
            errors += [f"[{i}]: {e}" for e in matches_schema(v, schema["items"])]
    return errors


def lint_tool(tool: dict) -> list[str]:
    problems = []
    if not NAME_RE.match(tool.get("name", "")):
        problems.append("name must match ^[a-zA-Z0-9_-]{1,128}$")
    desc = tool.get("description", "")
    if desc.count(". ") + desc.endswith(".") < 3:
        problems.append("description has fewer than 3 sentences (aim for 3-4+)")
    if not re.search(r"\b(do not|don't|not for|instead)\b", desc, re.I):
        problems.append("description never says when NOT to use the tool")
    for ex in tool.get("input_examples", []):
        errs = matches_schema(ex, tool["input_schema"])
        if errs:
            problems.append(f"input_example {ex} is invalid: {errs}")
    return problems


MINIMAL_TOOLS = [
    {"name": "get_customer", "description": "Gets customer info.", "input_schema": TOOLS_BY_NAME["get_customer"]["input_schema"]},
    {"name": "lookup_order", "description": "Gets order info.", "input_schema": TOOLS_BY_NAME["lookup_order"]["input_schema"]},
    {"name": "process_refund", "description": "Refunds an order.", "input_schema": TOOLS_BY_NAME["process_refund"]["input_schema"]},
]
bad_example = {**TOOLS_BY_NAME["get_customer"], "input_examples": [{"customer_id": "48213"}]}

for t in MINIMAL_TOOLS + RICH_TOOLS + [bad_example]:
    print(f"{t['name']:15} {'rich' if t in RICH_TOOLS else '----'} -> {lint_tool(t) or 'OK'}")

> **Observe:** the minimal one-liners fail the lint, and the `input_examples` entry with `"48213"` (a string for an integer field) is caught before it can cost you a 400.

> **Try it:** write a fourth tool, `get_customer_orders`, whose description overlaps with `get_customer`. Then ask yourself which one Claude should pick for "what did Jane buy?" If you can't answer in one second, neither can the model: fix the descriptions, not the prompt.

### Experiment: minimal vs. rich descriptions (live)

The README planned a 20-prompt selection test. Here are 8 prompts with the expected **first** tool; add your own to grow it to 20. It costs about 16 small calls.

In [ ]:
SELECTION_CASES = [
    ("What's the status of ORD-5002?", "lookup_order"),
    ("Is jane@example.com's account still active?", "get_customer"),
    ("What did I buy in order ORD-5001?", "lookup_order"),
    ("Customer 51877 says they can't log in. What's their account status?", "get_customer"),
    ("I'm jane@example.com and my mug arrived broken, I want my money back.", "get_customer"),
    ("How much did order ORD-5003 cost?", "lookup_order"),
    ("Which orders does omar@example.com have?", "get_customer"),
    ("Has ORD-5001 already been partly refunded?", "lookup_order"),
]


def first_tool(tools: list[dict], prompt: str, llm: anthropic.Anthropic = client) -> str:
    r = llm.messages.create(model=MODEL, max_tokens=1024, tools=tools,
                            messages=[{"role": "user", "content": prompt}])
    return next((b.name for b in r.content if b.type == "tool_use"), "(no tool)")


def selection_table(results: dict[str, list[str]]) -> None:
    print(f"{'prompt':55} {'expected':14} " + " ".join(f"{k:14}" for k in results))
    for i, (prompt, expected) in enumerate(SELECTION_CASES):
        row = " ".join(f"{('✓ ' if results[k][i] == expected else '✗ ') + results[k][i]:14}" for k in results)
        print(f"{prompt[:54]:55} {expected:14} {row}")
    for k, picks in results.items():
        acc = sum(p == e for p, (_, e) in zip(picks, SELECTION_CASES)) / len(SELECTION_CASES)
        print(f"accuracy[{k}] = {acc:.0%}")


results = {
    "minimal": [first_tool(MINIMAL_TOOLS, p) for p, _ in SELECTION_CASES],
    "rich": [first_tool(RICH_TOOLS, p) for p, _ in SELECTION_CASES],
}
selection_table(results)

> **Observe:** strong models often get easy prompts right even with poor descriptions; the gap shows up on ambiguous prompts (the refund request should start with *verification*, which only the rich `get_customer` description asks for). Look at *which* prompts flip, not only the totals.

> **Try it:** add a system prompt line such as "Always look up the account first when the user mentions their account." and rerun. Section 2.2 of the README warns that system-prompt wording can override otherwise clear descriptions.

### Scope, split and constrain the toolset (offline)

Four tool-set design moves from the README, as code you can check:

1. **Scope by role.** Each agent gets only its role's tools, plus at most one narrow cross-role tool (`verify_fact` for the synthesizer). New research goes back through the coordinator.
2. **Split a generic tool** (`analyze_document`) into purpose-specific tools with their own input/output contracts.
3. **Constrain a generic tool.** `load_document` replaces `fetch_url` and rejects anything that is not a document on an allowed host, **in code**.
4. **Audit the system prompt** for keyword-triggered rules that name a tool and can override good descriptions.

In [ ]:
import urllib.parse

ROLE_TOOLS = {
    "web_researcher": ["web_search", "load_document"],
    "document_analyst": ["extract_data_points", "summarize_content", "verify_claim_against_source"],
    "synthesizer": ["verify_fact"],  # scoped cross-role tool; anything bigger is routed via the coordinator
}
SPLIT_CONTRACTS = {  # what replaced the generic analyze_document
    "extract_data_points": "document_id -> list of {field, value, page}",
    "summarize_content": "document_id, max_words -> summary text",
    "verify_claim_against_source": "claim, document_id -> {verdict: supported|unsupported|unclear, evidence}",
}
ALLOWED_DOC_HOSTS = {"docs.example.com", "files.example.com"}
ALLOWED_DOC_TYPES = (".pdf", ".docx", ".md", ".txt")


def load_document(url: str) -> dict:
    """Constrained alternative to a generic fetch_url: only documents from allowed hosts."""
    parsed = urllib.parse.urlparse(url)
    if parsed.scheme != "https" or parsed.hostname not in ALLOWED_DOC_HOSTS:
        raise ValueError(f"{url} is not on an allowed document host {sorted(ALLOWED_DOC_HOSTS)}. "
                         "Use web_search results only through the researcher agent.")
    if not parsed.path.lower().endswith(ALLOWED_DOC_TYPES):
        raise ValueError(f"{parsed.path} is not a document ({', '.join(ALLOWED_DOC_TYPES)}).")
    return {"url": url, "status": "loaded (stub)"}


TRIGGER = re.compile(r"\b(whenever|any ?time|every time|if the user (?:mentions|says|writes))\b", re.I)


def keyword_tool_rules(system_prompt: str, tool_names: list[str]) -> list[str]:
    """Sentences that tie a keyword trigger to a specific tool: candidates for unintended routing."""
    sentences = re.split(r"(?<=[.!?])\s+", system_prompt)
    return [s for s in sentences if TRIGGER.search(s) and any(n in s for n in tool_names)]


assert "web_search" not in ROLE_TOOLS["synthesizer"], "the synthesizer must not research on its own"
for url in ["https://docs.example.com/q3-report.pdf", "https://evil.example.net/x.pdf", "https://docs.example.com/admin"]:
    try:
        print("load_document OK     :", load_document(url))
    except ValueError as e:
        print("load_document rejects:", e)

risky_system = ("You are a support agent. Whenever the user mentions their account, call get_customer first. "
                "Be concise. Verify identity before any refund.")
print("\nkeyword-triggered rules to review:", keyword_tool_rules(risky_system, list(TOOLS_BY_NAME)))

> **Observe:** `load_document` turns "please only fetch documents" from a hope into a guarantee, and its errors say what to do instead. The audit flags the "whenever the user mentions their account" rule: "Which account was ORD-5001 billed to?" would now go to `get_customer` instead of `lookup_order`. "Verify identity before any refund" is not flagged: it is a condition on intent, not on a keyword.

> **Try it:** rewrite `risky_system` so the audit returns `[]` while keeping the business rule (hint: state the condition as an intent, such as "before discussing or refunding an order").

### Bonus: schemas from Python functions (offline)

The SDK's beta Tool Runner builds the tool definition from a typed function and its docstring. Useful to know what it generates even if you write the loop yourself.

In [ ]:
@anthropic.beta_tool
def lookup_order_fn(order_id: str) -> str:
    """Fetch one order by its order ID (format ORD-####). Use for status, contents or price questions.

    Args:
        order_id: Order ID, e.g. ORD-5001.
    """
    return json.dumps(ORDERS.get(order_id, {}))


print(json.dumps(lookup_order_fn.to_dict(), indent=1))

## 2.3 The `tool_choice` parameter

`auto` (Claude decides), `any` (must call some tool), `tool` (must call this tool), `none` (no tools). `disable_parallel_tool_use: true` goes **inside** `tool_choice`. Claude Opus 5.5, Sonnet 5.5 and Fable 5.1 reject `any`/`tool` with a 400, so on those models you use `auto` + `strict: true` + a code check. [Theory](README.md#23-the-tool_choice-parameter)

In [ ]:
TOOL_CHOICES = {
    "auto": {"type": "auto"},
    "any": {"type": "any"},
    "tool": {"type": "tool", "name": "lookup_order"},
    "none": {"type": "none"},
    "auto, at most one call": {"type": "auto", "disable_parallel_tool_use": True},
}
REJECTS_FORCED = {"claude-opus-5-5", "claude-sonnet-5-5", "claude-fable-5-1", "claude-mythos-5-1"}

for label, choice in TOOL_CHOICES.items():
    forced = choice["type"] in ("any", "tool")
    verdict = "400 on this model" if forced and MODEL in REJECTS_FORCED else "accepted"
    print(f"{label:24} {json.dumps(choice):70} on {MODEL}: {verdict}")

### Compare the four modes (live, on a model that supports forced tool use)

The prompt is a greeting, so a sensible model would *not* call a tool. Watch what each mode does with it.

In [ ]:
prompt = [{"role": "user", "content": "Hi! Quick question before I order: do you ship to Canada?"}]
for label in ("auto", "any", "tool", "none"):
    r = client.messages.create(model=FORCED_TOOL_MODEL, max_tokens=512, tools=RICH_TOOLS,
                               tool_choice=TOOL_CHOICES[label], messages=prompt)
    kinds = [b.type if b.type != "tool_use" else f"tool_use:{b.name}" for b in r.content]
    print(f"{label:5} stop_reason={r.stop_reason:9} content={kinds}")

> **Observe:** `any` and `tool` force a call even though it makes no sense here, and they emit **no text before** the `tool_use` block (the API prefills the assistant turn). `none` answers in text even though the tools are in the request.

### Reproduce the 400 on the default model (live)

In [ ]:
try:
    client.messages.create(model=MODEL, max_tokens=256, tools=RICH_TOOLS,
                           tool_choice={"type": "any"},
                           messages=[{"role": "user", "content": "Where is ORD-5001?"}])
    print("accepted (this model supports forced tool use)")
except anthropic.BadRequestError as e:
    print("400 BadRequestError:", e.message)

### `require_tool_call()`: auto + strict + verify (offline test, then live)

The guarantee `any`/`tool` used to give now comes from code: ask for the tool with `auto`, keep schema-valid arguments with `strict: true`, and **check** that the call happened; nudge and retry if it didn't.

In [ ]:
def require_tool_call(llm: anthropic.Anthropic, messages: list, tools: list[dict], expected: str,
                      retries: int = 2, model: str = MODEL) -> dict:
    """Ask for a specific tool with tool_choice=auto and verify the call actually happened."""
    for _ in range(retries + 1):
        response = llm.messages.create(
            model=model, max_tokens=1024, messages=messages, tool_choice={"type": "auto"},
            tools=[{**t, "strict": True} for t in tools],  # schemas already set additionalProperties: false
        )
        for block in response.content:
            if block.type == "tool_use" and block.name == expected:
                return block.input
        if response.stop_reason == "tool_use":
            raise RuntimeError("Model called a different tool; route through the normal tool loop")
        if response.stop_reason in ("refusal", "max_tokens"):
            raise RuntimeError(f"Stopped with {response.stop_reason}; handle it explicitly")
        messages = messages + [
            {"role": "assistant", "content": response.content},
            {"role": "user", "content": f"Please call the {expected} tool now."},
        ]
    raise RuntimeError(f"{expected} was not called after {retries + 1} attempts")


fake = ScriptedClaude(
    reply(text("Sure, which order do you mean?")),                      # 1st try: plain text, no call
    reply(tool_call("lookup_order", {"order_id": "ORD-5001"})),          # 2nd try: the call we want
)
args = require_tool_call(fake.client, [{"role": "user", "content": "Check ORD-5001"}], RICH_TOOLS, "lookup_order")
print("got arguments:", args)
print("nudge we sent:", fake.requests[1]["messages"][-1])
print("strict flag on the wire:", [t.get("strict") for t in fake.requests[0]["tools"]])

In [ ]:
# (live) the same helper on the real default model
print(require_tool_call(client, [{"role": "user", "content": "Use lookup_order to check ORD-5003."}],
                        RICH_TOOLS, "lookup_order"))

> **Try it:** script three plain-text replies in the offline cell and confirm that `require_tool_call` raises after `retries + 1` attempts instead of looping forever.

### Force the first step, then continue with `auto` (offline)

Forced selection guarantees only **one** step. The exam pattern: force `extract_metadata` on turn 1 (on a model that accepts forced tool use), then switch `tool_choice` back to `auto` so Claude picks the enrichment tools in follow-up turns.

In [ ]:
PIPELINE_TOOLS = [
    {"name": "extract_metadata", "description": "Extract title, parties and effective date from a contract. Call first.",
     "input_schema": {"type": "object", "properties": {
         "title": {"type": "string"}, "parties": {"type": "array", "items": {"type": "string"}},
         "effective_date": {"type": ["string", "null"]}},
         "required": ["title", "parties", "effective_date"], "additionalProperties": False}},
    {"name": "enrich_party", "description": "Look up one contract party in the vendor registry by exact legal name.",
     "input_schema": {"type": "object", "properties": {"legal_name": {"type": "string"}},
                      "required": ["legal_name"], "additionalProperties": False}},
]

fake = ScriptedClaude(
    reply(tool_call("extract_metadata", {"title": "MSA", "parties": ["Acme Ltd", "Globex BV"], "effective_date": None})),
    reply(tool_call("enrich_party", {"legal_name": "Acme Ltd"}), tool_call("enrich_party", {"legal_name": "Globex BV"})),
)
msgs = [{"role": "user", "content": "<contract text>"}]
step1 = fake.client.messages.create(model=FORCED_TOOL_MODEL, max_tokens=1024, tools=PIPELINE_TOOLS,
                                    tool_choice={"type": "tool", "name": "extract_metadata"}, messages=msgs)
meta = next(b for b in step1.content if b.type == "tool_use")
msgs += [{"role": "assistant", "content": step1.content},
         {"role": "user", "content": [{"type": "tool_result", "tool_use_id": meta.id, "content": "stored"}]}]
step2 = fake.client.messages.create(model=FORCED_TOOL_MODEL, max_tokens=1024, tools=PIPELINE_TOOLS,
                                    tool_choice={"type": "auto"}, messages=msgs)
for i, req in enumerate(fake.requests, 1):
    print(f"request {i}: tool_choice={req['tool_choice']}")
show(step2)

> **Observe:** request 1 pins the first step; request 2 hands control back to the model, which fans out to two parallel `enrich_party` calls based on what step 1 extracted. On Opus 5.5 / Sonnet 5.5 / Fable 5.1 you would replace step 1 with `require_tool_call()` or with `messages.parse()` for the metadata.

## 2.4 JSON schemas for structured output

Three mechanisms: a plain tool schema (usually right, not guaranteed), `strict: true` on a tool (arguments always validate), and JSON outputs via `output_config.format` / `client.messages.parse(output_format=PydanticModel)` (the text answer always validates). Design the schema honestly: required + nullable for maybe-missing data, enums with `"other"` + detail and `"unclear"`. [Theory](README.md#24-json-schemas-for-structured-output)

### One contract, three request shapes (offline)

The same ticket-classification contract written as a plain tool, a strict tool, and a Pydantic model for `messages.parse()`.

In [ ]:
TICKET_SCHEMA = {
    "type": "object",
    "properties": {
        "category": {"type": "string", "enum": ["bug", "feature", "docs", "unclear", "other"]},
        "category_detail": {"type": ["string", "null"],
                            "description": "Short explanation when category is 'other' or 'unclear'; otherwise null."},
        "severity": {"type": "string", "enum": ["critical", "high", "medium", "low"]},
        "customer_email": {"type": ["string", "null"],
                           "description": "Email exactly as written in the ticket; null if the ticket has none."},
    },
    "required": ["category", "category_detail", "severity", "customer_email"],
    "additionalProperties": False,
}
plain_tool = {"name": "record_ticket_classification",
              "description": "Record the classification of one support ticket. Call exactly once per ticket.",
              "input_schema": TICKET_SCHEMA}
strict_tool = {**plain_tool, "strict": True}


def lower_if_str(v: Any) -> Any:
    # Structured outputs don't guarantee enum letter-casing ("Bug" for "bug"), so normalize before validation.
    return v.lower() if isinstance(v, str) else v


Category = Annotated[Literal["bug", "feature", "docs", "unclear", "other"], BeforeValidator(lower_if_str)]
Severity = Annotated[Literal["critical", "high", "medium", "low"], BeforeValidator(lower_if_str)]


class TicketClassification(BaseModel):
    category: Category
    category_detail: Optional[str]  # required key, value may be None
    severity: Severity
    customer_email: Optional[str]


# What messages.parse() will send as output_config.format:
print(json.dumps(anthropic.transform_schema(TicketClassification), indent=1))

> **Observe:** `transform_schema` (the function `messages.parse()` uses) adds `additionalProperties: false`, keeps every field in `required`, and turns `Optional[str]` into `anyOf [string, null]`: the "required + nullable" pattern for free.

### A strict-schema linter (offline)

Strict mode and JSON outputs accept only a subset of JSON Schema and have per-request complexity limits. This checker catches the common mistakes before the API does.

In [ ]:
UNSUPPORTED = {"minimum", "maximum", "exclusiveMinimum", "exclusiveMaximum", "multipleOf",
               "minLength", "maxLength", "maxItems", "uniqueItems"}
LIMITS = {"strict_tools": 20, "optional_params": 24, "union_params": 16}


def strict_schema_report(schema: dict, path: str = "$") -> dict:
    """Return {'problems': [...], 'optional': n, 'unions': n} for one strict schema."""
    report = {"problems": [], "optional": 0, "unions": 0}
    for key in UNSUPPORTED & schema.keys():
        report["problems"].append(f"{path}: '{key}' is not supported in strict schemas; validate it in code")
    if schema.get("minItems", 0) > 1:
        report["problems"].append(f"{path}: minItems > 1 is not supported")
    if schema.get("type") == "object" or "properties" in schema:
        if schema.get("additionalProperties") is not False:
            report["problems"].append(f"{path}: objects need additionalProperties: false")
        required = set(schema.get("required", []))
        for name, sub in schema.get("properties", {}).items():
            report["optional"] += name not in required
            report["unions"] += isinstance(sub.get("type"), list) or "anyOf" in sub
            child = strict_schema_report(sub, f"{path}.{name}")
            for k in report:
                report[k] += child[k]
    if "items" in schema:
        child = strict_schema_report(schema["items"], f"{path}[]")
        for k in report:
            report[k] += child[k]
    return report


bad_schema = {
    "type": "object",
    "properties": {
        "category": {"type": "string", "enum": ["bug", "other"]},
        "confidence": {"type": "number", "minimum": 0, "maximum": 1},
        "tags": {"type": "array", "items": {"type": "string"}, "maxItems": 5},
        "contact": {"type": "object", "properties": {"email": {"type": "string"}}},
    },
    "required": ["category"],
}
for label, schema in [("TICKET_SCHEMA", TICKET_SCHEMA), ("bad_schema", bad_schema)]:
    rep = strict_schema_report(schema)
    print(f"{label}: optional={rep['optional']}/{LIMITS['optional_params']} unions={rep['unions']}/{LIMITS['union_params']}")
    for p in rep["problems"]:
        print("   -", p)

> **Try it:** count the optional and union parameters across *all three* strict support tools (`RICH_TOOLS`) plus `TICKET_SCHEMA`. The limits (24 optional, 16 unions, 20 strict tools) are summed over every strict schema in one request.

### Normalization rules and flexible attributes (offline)

Two schema-design patterns for messy sources:

- **Normalization rules in the prompt**, next to a strict schema: the schema fixes the *shape*, the rules fix the *format* (dates, money, units), and a small code check confirms both.
- **Flexible extraction** for fields you can't list in advance: a strict-compatible **key/value list**, versus an open object (`additionalProperties: true`) that strict mode rejects.

In [ ]:
NORMALIZATION_RULES = """Normalize before you fill the schema:
- Dates -> ISO 8601 (YYYY-MM-DD). UK and EU suppliers write day/month/year.
- Money -> integer cents in the listed currency ("1.299,00 EUR" -> 129900).
- If a value is missing or unreadable, use null. Never guess."""

SPEC_SHEET_TOOL = {
    "name": "record_spec_sheet", "strict": True,
    "description": "Record one product spec sheet. Known fields are typed; anything else goes in attributes.",
    "input_schema": {
        "type": "object",
        "properties": {
            "product_name": {"type": "string"},
            "price_cents": {"type": ["integer", "null"], "description": "Integer cents; null if no price is printed."},
            "release_date": {"type": ["string", "null"], "description": "ISO 8601 date (YYYY-MM-DD) or null."},
            "attributes": {   # flexible, but every entry has a guaranteed shape
                "type": "array",
                "items": {"type": "object",
                          "properties": {"name": {"type": "string"}, "value": {"type": "string"},
                                         "unit": {"type": ["string", "null"]}},
                          "required": ["name", "value", "unit"], "additionalProperties": False},
            },
        },
        "required": ["product_name", "price_cents", "release_date", "attributes"],
        "additionalProperties": False,
    },
}
OPEN_OBJECT_VARIANT = {**SPEC_SHEET_TOOL["input_schema"],
                       "properties": {**SPEC_SHEET_TOOL["input_schema"]["properties"],
                                      "attributes": {"type": "object", "additionalProperties": True}}}

ISO_DATE = re.compile(r"^\d{4}-\d{2}-\d{2}$")


def normalization_errors(record: dict) -> list[str]:
    errs = []
    if record["release_date"] is not None and not ISO_DATE.match(record["release_date"]):
        errs.append(f"release_date {record['release_date']!r} is not YYYY-MM-DD")
    if record["price_cents"] is not None and record["price_cents"] <= 0:
        errs.append("price_cents must be positive")
    return errs


for label, schema in [("key/value list", SPEC_SHEET_TOOL["input_schema"]), ("open object", OPEN_OBJECT_VARIANT)]:
    print(f"{label:15} strict problems -> {strict_schema_report(schema)['problems'] or 'none'}")

fake = ScriptedClaude(reply(tool_call("record_spec_sheet", {
    "product_name": "Kettle K2", "price_cents": 4999, "release_date": "03/04/2026",
    "attributes": [{"name": "capacity", "value": "1.7", "unit": "l"}, {"name": "colour", "value": "steel", "unit": None}]})))
r = fake.client.messages.create(model=MODEL, max_tokens=1024, system=NORMALIZATION_RULES, tools=[SPEC_SHEET_TOOL],
                                messages=[{"role": "user", "content": "<spec sheet text>"}])
record = next(b for b in r.content if b.type == "tool_use").input
print("attributes as a dict:", {a["name"]: (a["value"], a["unit"]) for a in record["attributes"]})
print("normalization check :", normalization_errors(record) or "OK")

> **Observe:** the key/value list passes the strict linter; the open object fails it (`additionalProperties: false` is required on every object). The scripted model ignored the date rule (`03/04/2026`), and the code check caught it. The rules make output consistent, the schema makes it well-formed, and only code confirms it.

### `messages.parse()` end to end (offline)

In [ ]:
fake = ScriptedClaude(reply(text(json.dumps(
    {"category": "Other", "category_detail": "asks to delete their account", "severity": "low", "customer_email": None}
))))
parsed = fake.client.messages.parse(
    model=MODEL, max_tokens=1024, output_format=TicketClassification,
    messages=[{"role": "user", "content": "Classify this support ticket:\n\nPlease delete my account."}],
)
print("parsed_output:", repr(parsed.parsed_output))
print("sent output_config.format.type:", fake.requests[0]["output_config"]["format"]["type"])

> **Observe:** the scripted model answered `"Other"` (capital O). Without the `BeforeValidator`, Pydantic would reject it; the docs warn that enum casing is not guaranteed, so compare enums case-insensitively.

### Classify real tickets: strict tool vs. `parse()` (live)

In [ ]:
TICKETS = [
    "The export button crashes the app every time since yesterday's update. Contact: lee@corp.io",
    "Could you add dark mode? Would be lovely.",
    "I want to change the legal entity name printed on all my invoices.",
]
for ticket in TICKETS:
    msgs = [{"role": "user", "content": f"Classify this support ticket with record_ticket_classification:\n\n{ticket}"}]
    via_tool = require_tool_call(client, msgs, [plain_tool], "record_ticket_classification")
    via_parse = client.messages.parse(
        model=MODEL, max_tokens=1024, output_format=TicketClassification,
        messages=[{"role": "user", "content": f"Classify this support ticket:\n\n{ticket}"}],
    ).parsed_output
    print(ticket[:60], "\n   strict tool:", via_tool, "\n   parse():    ", via_parse.model_dump())

> **Observe:** the missing email comes back as `null` instead of an invented address, and the invoice-name request should land in `"other"` with a useful `category_detail`. That is the schema design doing its job.

## 2.5 Syntax vs. semantic errors

Strict schemas and JSON outputs remove **syntax** errors (bad JSON, wrong types, missing keys), except for `max_tokens` truncation, refusals and enum casing. **Semantic** errors (totals that don't add up, invented IDs) need your own validation, and a retry with *specific* feedback only helps when the answer is actually in the source. [Theory](README.md#25-syntax-vs-semantic-errors)

In [ ]:
class LineItem(BaseModel):
    description: str
    amount_cents: int


class InvoiceExtraction(BaseModel):
    invoice_number: Optional[str]   # null when not printed on the invoice
    line_items: list[LineItem]
    stated_total_cents: int         # the total exactly as printed


def semantic_errors(inv: InvoiceExtraction) -> list[str]:
    errors = []
    calculated = sum(i.amount_cents for i in inv.line_items)
    if calculated != inv.stated_total_cents:
        errors.append(f"sum(line_items) = {calculated} but stated_total_cents = {inv.stated_total_cents}. "
                      "Re-check each line amount against the document.")
    if any(i.amount_cents < 0 for i in inv.line_items):
        errors.append("Negative line amount; discounts must be their own line with a description.")
    return errors


class NeedsHumanReview(Exception):
    pass


def extract_with_retry(llm: anthropic.Anthropic, document: str, max_retries: int = 2) -> InvoiceExtraction:
    messages = [{"role": "user", "content": f"Extract the invoice data. Amounts in cents.\n\n{document}"}]
    errors: list[str] = []
    for _ in range(max_retries + 1):
        try:
            response = llm.messages.parse(model=MODEL, max_tokens=2048, messages=messages,
                                          output_format=InvoiceExtraction)
        except ValidationError as exc:      # parse() raises when the text isn't valid JSON (e.g. truncated)
            raise NeedsHumanReview(f"Unparseable output: {exc.errors()[0]['msg']}") from exc
        if response.stop_reason in ("refusal", "max_tokens") or response.parsed_output is None:
            raise NeedsHumanReview(f"No valid extraction (stop_reason={response.stop_reason})")
        invoice = response.parsed_output
        errors = semantic_errors(invoice)
        if not errors:
            return invoice
        messages += [
            {"role": "assistant", "content": response.content},
            {"role": "user", "content": "Validation failed:\n- " + "\n- ".join(errors)
             + "\nCorrect only the affected fields. If a value is truly not in the document, use null."},
        ]
    raise NeedsHumanReview(f"Still inconsistent after retries: {errors}")


INVOICE = """ACME Supplies. Invoice INV-2207
2 x Widget ........ $40.00
1 x Shipping ...... $ 7.50
TOTAL DUE ......... $47.50"""
print("defined; sample invoice:\n" + INVOICE)

### Three offline scenarios

1. A semantic slip that a retry fixes. 2. Data that is not in the source (accept `null`, don't retry). 3. A truncated reply (`max_tokens`): `parse()` raises before you ever see `parsed_output`.

In [ ]:
wrong = {"invoice_number": "INV-2207", "stated_total_cents": 4750,
         "line_items": [{"description": "2 x Widget", "amount_cents": 4000}, {"description": "Shipping", "amount_cents": 570}]}
fixed = {**wrong, "line_items": [{"description": "2 x Widget", "amount_cents": 4000},
                                 {"description": "Shipping", "amount_cents": 750}]}

# 1) semantic slip -> retry with specific feedback
fake = ScriptedClaude(reply(text(json.dumps(wrong))), reply(text(json.dumps(fixed))))
inv = extract_with_retry(fake.client, INVOICE)
print("1) accepted after", len(fake.requests), "calls:", inv.line_items[1])
print("   feedback sent:", fake.requests[1]["messages"][-1]["content"].splitlines()[1])

# 2) invoice number genuinely absent -> null is a valid answer, no retry needed
no_number = {**fixed, "invoice_number": None}
fake = ScriptedClaude(reply(text(json.dumps(no_number))))
inv = extract_with_retry(fake.client, INVOICE)
print("2) accepted after", len(fake.requests), "call: invoice_number =", inv.invoice_number)

# 3) truncated output
fake = ScriptedClaude(reply(text('{"invoice_number": "INV-2207", "line_it'), stop_reason="max_tokens"))
try:
    extract_with_retry(fake.client, INVOICE)
except NeedsHumanReview as e:
    print("3) escalated:", e)

> **Observe:** the retry message names the exact discrepancy (`570` vs. `750`), so the model can fix one field instead of re-guessing everything. Scenario 3 shows a real SDK detail: `messages.parse()` raises `pydantic.ValidationError` on unparseable text, so the `stop_reason` check alone is not enough; catch the exception too.

> **Try it:** script three replies that are all wrong and confirm the function escalates with `NeedsHumanReview` after `max_retries + 1` calls instead of returning bad data.

In [ ]:
# (live) the real model on the sample invoice
inv = extract_with_retry(client, INVOICE)
print(inv.model_dump(), "| semantic errors:", semantic_errors(inv))

## 2.6 The client-side tool loop and `tool_result` rules

Call → check `stop_reason` → run **every** `tool_use` → send **all** results in **one** user message (results first, ids matched, failures as `is_error: true`) → repeat until `end_turn`, with `max_turns` as a safety net. Never execute tool calls from a `max_tokens` or `refusal` turn. [Theory](README.md#26-the-client-side-tool-loop-and-tool_result-rules-expanded-beyond-the-guide)

### A protocol validator for the 400 rules (offline)

In [ ]:
def _get(block: Any, key: str) -> Any:
    return block.get(key) if isinstance(block, dict) else getattr(block, key, None)


def protocol_errors(messages: list[dict]) -> list[str]:
    """Check the tool_use/tool_result pairing rules that otherwise come back as HTTP 400."""
    errors = []
    for i, msg in enumerate(messages):
        if msg["role"] != "assistant" or isinstance(msg["content"], str):
            continue
        ids = [_get(b, "id") for b in msg["content"] if _get(b, "type") == "tool_use"]
        if not ids:
            continue
        nxt = messages[i + 1] if i + 1 < len(messages) else None
        if nxt is None:
            continue  # pending: we are about to send the results
        if nxt["role"] != "user" or isinstance(nxt["content"], str):
            errors.append(f"message {i + 1} must be a user message with tool_result blocks")
            continue
        types = [_get(b, "type") for b in nxt["content"]]
        n_results = sum(t == "tool_result" for t in types)
        if "tool_result" in types and types[:n_results] != ["tool_result"] * n_results:
            errors.append(f"message {i + 1}: tool_result blocks must come first (text only after them)")
        answered = {_get(b, "tool_use_id") for b in nxt["content"] if _get(b, "type") == "tool_result"}
        missing = [x for x in ids if x not in answered]
        if missing:
            errors.append(f"message {i + 1}: no tool_result for {missing} immediately after the tool_use")
    return errors


assistant = {"role": "assistant", "content": [tool_call("lookup_order", {"order_id": "ORD-5001"}, id="t1"),
                                              tool_call("lookup_order", {"order_id": "ORD-5002"}, id="t2")]}
res = lambda i: {"type": "tool_result", "tool_use_id": i, "content": "{}"}
cases = {
    "correct (one message, results first)": [assistant, {"role": "user", "content": [res("t1"), res("t2"), text("thanks")]}],
    "text before results": [assistant, {"role": "user", "content": [text("Here you go"), res("t1"), res("t2")]}],
    "results split over two messages": [assistant, {"role": "user", "content": [res("t1")]},
                                        {"role": "user", "content": [res("t2")]}],
    "message inserted in between": [assistant, {"role": "user", "content": "hello?"},
                                    {"role": "user", "content": [res("t1"), res("t2")]}],
}
for label, history in cases.items():
    print(f"{label:38} -> {protocol_errors([{'role': 'user', 'content': 'q'}] + history) or 'OK'}")

> **Observe:** splitting parallel results across messages is a protocol error *and* a behavioral one: the docs warn it teaches Claude to stop making parallel calls.

### A generic, bounded loop with `is_error` handling (offline)

In [ ]:
class ToolInputError(Exception):
    """The arguments are wrong; retrying the same call will never work."""


def lookup_order_tool(order_id: str) -> dict:
    if not re.fullmatch(r"ORD-\d{4}", order_id):
        raise ToolInputError(f"order_id '{order_id}' is malformed; expected ORD-#### such as ORD-5001")
    if order_id not in ORDERS:
        raise ToolInputError(f"Order {order_id} not found. Ask the customer to confirm the order ID.")
    return {k: ORDERS[order_id][k] for k in ("order_id", "status", "total_cents", "refunded_cents")}


def run_agent(llm: anthropic.Anthropic, user_text: str, tools: list[dict], handlers: dict[str, Callable],
              max_turns: int = 10, model: str = MODEL):
    messages: list[dict] = [{"role": "user", "content": user_text}]
    for _ in range(max_turns):                                   # safety net, not the exit condition
        response = llm.messages.create(model=model, max_tokens=1024, tools=tools, messages=messages)
        messages.append({"role": "assistant", "content": response.content})   # FULL content
        if response.stop_reason != "tool_use":                   # end_turn, max_tokens, refusal, ...
            return response, messages
        results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            try:
                if block.name not in handlers:
                    raise ToolInputError(f"Unknown tool {block.name}")
                output = handlers[block.name](**block.input)
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(output)})
            except ToolInputError as exc:                        # actionable message, not "failed"
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(exc), "is_error": True})
            except Exception as exc:                             # unexpected backend failure
                results.append({"type": "tool_result", "tool_use_id": block.id,
                                "content": f"{type(exc).__name__}: {exc}", "is_error": True})
        messages.append({"role": "user", "content": results})    # ALL results, ONE message
    raise RuntimeError("Agent exceeded max_turns")


fake = ScriptedClaude(
    reply(tool_call("lookup_order", {"order_id": "ORD-5001"}), tool_call("lookup_order", {"order_id": "ORD-9999"})),
    reply(text("ORD-5001 is delivered. I couldn't find ORD-9999; could you double-check that number?")),
)
final, history = run_agent(fake.client, "Status of ORD-5001 and ORD-9999?", RICH_TOOLS[1:2],
                           {"lookup_order": lookup_order_tool})
show(final)
print("\ntool_result message we sent:")
print(json.dumps(fake.requests[1]["messages"][-1], indent=1))
print("protocol errors in final history:", protocol_errors(history) or "none")

> **Observe:** two parallel `tool_use` blocks produced one user message with two `tool_result` blocks, and the unknown order came back as `is_error: true` with a message that tells the model what to do next.

### Transient vs. validation errors (offline)

In [ ]:
_flaky_calls = {"n": 0}


def flaky_backend(order_id: str) -> dict:
    """Simulates a backend that times out on the first two calls."""
    _flaky_calls["n"] += 1
    if _flaky_calls["n"] <= 2:
        raise TimeoutError("orders-db timed out")
    return lookup_order_tool(order_id)


def with_backoff(fn: Callable, attempts: int = 3, base_delay: float = 0.05) -> Callable:
    """Retry TRANSIENT failures inside the tool; let validation errors through immediately."""
    def wrapper(**kwargs):
        for attempt in range(attempts):
            try:
                return fn(**kwargs)
            except TimeoutError:
                if attempt == attempts - 1:
                    raise ToolInputError("Order service unavailable after 3 attempts. Retry in a minute.")
                time.sleep(base_delay * 2 ** attempt)
    return wrapper


robust_lookup = with_backoff(flaky_backend)
print("transient (retried inside the tool):", robust_lookup(order_id="ORD-5001"), "after", _flaky_calls["n"], "backend calls")
try:
    robust_lookup(order_id="5001")
except ToolInputError as e:
    print("validation (returned at once):", e, "| backend calls:", _flaky_calls["n"])

### Real parallel calls and a real 400 (live)

In [ ]:
final, history = run_agent(client, "What's the status of ORD-5001 and ORD-5002?", RICH_TOOLS[1:2],
                           {"lookup_order": lookup_order_tool})
show(final)
calls_per_turn = [sum(_get(b, "type") == "tool_use" for b in m["content"]) for m in history if m["role"] == "assistant"]
print("tool_use blocks per assistant turn:", calls_per_turn)

# Now break a rule on purpose: text BEFORE the tool_result blocks
broken = [m if i != 2 else {"role": "user", "content": [text("Here are the results:")] + list(m["content"])}
          for i, m in enumerate(history[:3])]
try:
    client.messages.create(model=MODEL, max_tokens=256, tools=RICH_TOOLS[1:2], messages=broken)
except anthropic.BadRequestError as e:
    print("400 as expected:", e.message[:200])

> **Try it:** remove one `tool_result` from `broken` instead of adding text, and compare the 400 message with what `protocol_errors()` predicted.

> **SDK shortcut:** the beta Tool Runner (`client.beta.messages.tool_runner(...)` with `@beta_tool` functions) runs this loop for you and still lets you inspect each turn. You wrote it by hand once so you know what it automates.

## 2.7 Client tools, Anthropic-schema tools and server tools

Three execution models share one `tools` array: **user-defined client tools** (you write and run them), **Anthropic-schema client tools** (`bash`, `str_replace_based_edit_tool`, the computer toolset: Anthropic publishes the schema, you run them), and **server tools** (`web_search`, `web_fetch`, `code_execution`: Anthropic runs them, you never send a `tool_result`). A harness is a loop plus a **dispatch table**. [Theory](README.md#27-client-tools-anthropic-schema-tools-and-server-tools-beyond-the-guide)

### Classify a mixed `tools` array (offline)

In [ ]:
SERVER_TYPES = ("web_search_", "web_fetch_", "code_execution_", "tool_search_tool_", "advisor_")
ANTHROPIC_CLIENT_TYPES = ("bash_", "text_editor_", "memory_", "computer_", "browser_")

MIXED_TOOLS = [
    TOOLS_BY_NAME["lookup_order"],                                              # user-defined
    {"type": "text_editor_20250728", "name": "str_replace_based_edit_tool", "max_characters": 10000},
    {"type": "computer_toolset_20260801", "configs": {"zoom": {"enabled": False}}},   # toolset: no name
    {"type": "web_search_20250305", "name": "web_search", "max_uses": 3,
     "allowed_domains": ["platform.claude.com", "docs.example.com"]},
]


def execution_model(tool: dict) -> str:
    t = tool.get("type", "custom")
    if t.startswith(SERVER_TYPES):
        return "server (Anthropic runs it; no tool_result)"
    if t.startswith(ANTHROPIC_CLIENT_TYPES):
        return "Anthropic-schema client (you run it)"
    return "user-defined client (you write and run it)"


for t in MIXED_TOOLS:
    print(f"{t.get('name', t.get('type')):30} -> {execution_model(t)}")

### A sandboxed text editor handler (offline)

The text editor tool's schema is Anthropic's; the **implementation and the safety rules are yours**: every path stays inside a sandbox root, `str_replace` must match exactly one location, and problems go back as `is_error` results with actionable text.

In [ ]:
import tempfile
from pathlib import Path

SANDBOX = Path(tempfile.mkdtemp(prefix="ch02_editor_"))


class EditorError(Exception):
    pass


def _safe(path: str) -> Path:
    p = (SANDBOX / path).resolve()
    if not p.is_relative_to(SANDBOX.resolve()):
        raise EditorError(f"Error: {path} is outside the allowed directory.")
    return p


def run_text_editor(command: str, path: str, **kw) -> str:
    p = _safe(path)
    if command == "view":
        if p.is_dir():
            return "\n".join(sorted(x.name for x in p.iterdir()))
        if not p.exists():
            raise EditorError(f"Error: File not found: {path}")
        lines = p.read_text().splitlines()
        start, end = kw.get("view_range") or (1, len(lines))
        end = len(lines) if end == -1 else end
        return "\n".join(f"{i}: {lines[i - 1]}" for i in range(start, end + 1))[:10000]   # max_characters
    if command == "create":
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(kw["file_text"])
        return f"Created {path}"
    if command == "str_replace":
        text_ = p.read_text()
        n = text_.count(kw["old_str"])
        if n == 0:
            raise EditorError("Error: No match found for replacement. Please check your text and try again.")
        if n > 1:
            raise EditorError(f"Error: Found {n} matches for replacement text. "
                              "Please provide more context to make a unique match.")
        p.write_text(text_.replace(kw["old_str"], kw["new_str"], 1))
        return "Successfully replaced text at exactly one location."
    if command == "insert":
        lines = p.read_text().splitlines()
        lines.insert(kw["insert_line"], kw["insert_text"])
        p.write_text("\n".join(lines) + "\n")
        return f"Inserted text after line {kw['insert_line']}"
    raise EditorError(f"Error: unknown command {command!r}")


print(run_text_editor("create", "notes.md", file_text="status: draft\nowner: ops\nstatus: draft\n"))
for cmd in [dict(command="str_replace", path="notes.md", old_str="status: draft", new_str="status: final"),
            dict(command="view", path="../etc/passwd")]:
    try:
        print(run_text_editor(**cmd))
    except EditorError as e:
        print("is_error ->", e)

### The harness: a dispatch table for every execution model (offline)

`Harness` dispatches client `tool_use` blocks by the **(`toolset_name`, `name`)** pair, never answers `server_tool_use` blocks, resends the assistant content on `pause_turn` (with a cap), and, on a mixed turn, answers only the client calls while a server call waits. Computer-use member calls in one turn form a **batch action**: run them in order and stop at the first failure.

In [ ]:
class Harness:
    def __init__(self, llm: anthropic.Anthropic, tools: list[dict], handlers: dict[tuple[str, str], Callable],
                 max_turns: int = 8, max_pauses: int = 3, model: str = MODEL):
        self.llm, self.tools, self.handlers = llm, tools, handlers
        self.max_turns, self.max_pauses, self.model = max_turns, max_pauses, model
        self.log: list[str] = []

    @staticmethod
    def pending_server_calls(content) -> list[str]:
        """server_tool_use ids that have no result block yet (a mixed turn waiting on your client tools)."""
        answered = {getattr(b, "tool_use_id", None) for b in content if b.type.endswith("_tool_result")}
        return [b.id for b in content if b.type == "server_tool_use" and b.id not in answered]

    def dispatch(self, content) -> list[dict]:
        results, failed_toolset = [], None
        for b in content:
            if b.type != "tool_use":          # text, thinking, server_tool_use, *_tool_result: nothing to run
                continue
            key = (b.toolset_name or "", b.name)
            res = {"type": "tool_result", "tool_use_id": b.id}
            if b.toolset_name:
                res["toolset_name"] = b.toolset_name   # member results echo the toolset name
            if failed_toolset and b.toolset_name == failed_toolset:   # batch action: stop at first failure
                results.append({**res, "is_error": True, "content": "Skipped: an earlier action in this batch failed."})
                continue
            try:
                handler = self.handlers[key]
            except KeyError:
                results.append({**res, "is_error": True, "content": f"Unknown tool {key}."})
                continue
            try:
                results.append({**res, "content": handler(**b.input)})
                self.log.append(f"ran {key}")
            except Exception as exc:  # noqa: BLE001 - every failure goes back to the model
                results.append({**res, "is_error": True, "content": str(exc)})
                self.log.append(f"FAILED {key}: {exc}")
                failed_toolset = b.toolset_name or failed_toolset
        return results

    def run(self, user_text: str):
        messages: list[dict] = [{"role": "user", "content": user_text}]
        pauses = 0
        for _ in range(self.max_turns):
            r = self.llm.messages.create(model=self.model, max_tokens=1024, tools=self.tools, messages=messages)
            if messages[-1]["role"] == "assistant":    # continuing a paused turn: keep roles alternating
                messages[-1] = {"role": "assistant", "content": list(messages[-1]["content"]) + list(r.content)}
            else:
                messages.append({"role": "assistant", "content": r.content})
            for b in r.content:
                if b.type == "server_tool_use":
                    self.log.append(f"server tool call {b.id}: {b.name}({json.dumps(b.input)})")
                elif b.type == "web_search_tool_result":
                    outcome = (f"{len(b.content)} results" if isinstance(b.content, list)
                               else f"error {b.content.error_code}")
                    self.log.append(f"  result for {b.tool_use_id}: {outcome}")
            if r.stop_reason == "pause_turn":
                pauses += 1
                if pauses > self.max_pauses:
                    return r, messages
                self.log.append("pause_turn: resending the assistant content as-is")
                continue                               # no tool_result, no 'continue' message
            if r.stop_reason != "tool_use":
                return r, messages
            waiting = self.pending_server_calls(r.content)
            if waiting:
                self.log.append(f"mixed turn: server call(s) {waiting} wait for our client results")
            messages.append({"role": "user", "content": self.dispatch(r.content)})   # only client results
        raise RuntimeError("Harness exceeded max_turns")

In [ ]:
def server_call(name: str, tool_input: dict, id: str) -> dict:
    return {"type": "server_tool_use", "id": id, "name": name, "input": tool_input}


def search_result(tool_use_id: str, *urls: str) -> dict:
    return {"type": "web_search_tool_result", "tool_use_id": tool_use_id,
            "content": [{"type": "web_search_result", "url": u, "title": u.split("/")[2],
                         "encrypted_content": "opaque", "page_age": "September 30, 2026"} for u in urls]}


def take_screenshot() -> str:
    return "[a real harness returns an image block here]"


def left_click(coordinate: list[int]) -> str:
    if coordinate[0] > 1920:
        raise RuntimeError(f"Coordinate {coordinate} is off-screen (display is 1920 px wide).")
    return "OK"


handlers = {
    ("", "lookup_order"): lambda order_id: json.dumps(lookup_order_tool(order_id)),
    ("", "str_replace_based_edit_tool"): run_text_editor,
    ("computer", "screenshot"): take_screenshot,
    ("computer", "left_click"): left_click,
}
fake = ScriptedClaude(
    # 1) web search ran on the server (call + result in the same turn) and Claude also wants a client tool
    reply(server_call("web_search", {"query": "tool_result ordering rules"}, "srvtoolu_01"),
          search_result("srvtoolu_01", "https://platform.claude.com/docs/en/x"),
          tool_call("str_replace_based_edit_tool", {"command": "create", "path": "research.md", "file_text": "- rule\n"})),
    # 2) the server loop hit its iteration limit
    reply(server_call("web_search", {"query": "pause_turn"}, "srvtoolu_02"), stop_reason="pause_turn"),
    # 3) the continued turn finishes the paused search, then a mixed turn: a server call WITHOUT its result + a client call
    reply(search_result("srvtoolu_02", "https://platform.claude.com/docs/en/y"),
          server_call("web_fetch", {"url": "https://platform.claude.com/docs/en/x"}, "srvtoolu_03"),
          tool_call("lookup_order", {"order_id": "ORD-5001"})),
    # 4) a computer-use batch action whose 2nd action fails
    reply({**tool_call("screenshot", {}), "toolset_name": "computer"},
          {**tool_call("left_click", {"coordinate": [9999, 10]}), "toolset_name": "computer"},
          {**tool_call("screenshot", {}), "toolset_name": "computer"}),
    reply(text("Done: notes saved, order checked.")),
)
harness = Harness(fake.client, MIXED_TOOLS, handlers)
final, history = harness.run("Research the tool_result rules, save notes, check ORD-5001, then click Save.")
print("\n".join(harness.log))
print("\nlast message of the request after pause_turn:", fake.requests[2]["messages"][-1]["role"])
print("results sent on the mixed turn:", [b["tool_use_id"] for b in fake.requests[3]["messages"][-1]["content"]])
print("batch results:", [(b.get("toolset_name"), b.get("is_error", False)) for b in fake.requests[4]["messages"][-1]["content"]])
print("same tools array on every request:", all(r["tools"] == fake.requests[0]["tools"] for r in fake.requests))
print("protocol errors:", protocol_errors(history) or "none")

> **Observe:**
> - No `tool_result` was ever sent for a `srvtoolu_…` id, not even on the mixed turn, where `web_fetch` was still waiting.
> - After `pause_turn`, the next request simply ends with the paused **assistant** message. When the continuation arrives, the harness merges it into that same assistant message so roles keep alternating (the docs' own example rebuilds the message list for the same reason).
> - The batch action ran in order: the screenshot ran, the failed click returned `is_error`, and the last screenshot was **skipped** with an error result (still one result per `tool_use`). Check the computer use docs for the exact skip text they define.
> - The `tools` array never changed: resuming a mixed turn without the waiting server tool returns a 400.

### Parse a server-tool result and its errors (offline)

Server-tool errors arrive inside a **200** response: `web_search_tool_result.content` is then an error object, not a list. Citations point back at the results.

In [ ]:
fake = ScriptedClaude(reply(
    server_call("web_search", {"query": "claude tool reference"}, "srvtoolu_10"),
    search_result("srvtoolu_10", "https://platform.claude.com/docs/en/agents-and-tools/tool-use/tool-reference"),
    server_call("web_search", {"query": "one search too many"}, "srvtoolu_11"),
    {"type": "web_search_tool_result", "tool_use_id": "srvtoolu_11",
     "content": {"type": "web_search_tool_result_error", "error_code": "max_uses_exceeded"}},
    {"type": "text", "text": "The tool reference lists every Anthropic-provided tool.",
     "citations": [{"type": "web_search_result_location", "url": "https://platform.claude.com/docs/en/agents-and-tools/tool-use/tool-reference",
                    "title": "Tool reference", "encrypted_index": "opaque", "cited_text": "Directory of Anthropic-provided..."}]},
    stop_reason="end_turn",
))
r = fake.client.messages.create(model=MODEL, max_tokens=1024, messages=[{"role": "user", "content": "q"}],
                                tools=[{"type": "web_search_20250305", "name": "web_search", "max_uses": 1}])
for b in r.content:
    if b.type == "web_search_tool_result":
        print(b.tool_use_id, "->", [x.url for x in b.content] if isinstance(b.content, list) else f"ERROR {b.content.error_code}")
    elif b.type == "text":
        print("text cites:", [c.url for c in (b.citations or [])])

### The "batch tool" pattern vs. native parallel calls (offline)

Older course material wraps several calls in one user-defined `batch_tool`. It works, but each inner call loses its own schema, `strict` guarantee and `tool_result`. Current models emit parallel `tool_use` blocks natively, so prefer those.

In [ ]:
BATCH_TOOL = {
    "name": "batch_tool",
    "description": "Run several independent tool calls at once. Use only for calls that do not depend on each other.",
    "input_schema": {"type": "object", "properties": {"invocations": {"type": "array", "items": {
        "type": "object", "properties": {"name": {"type": "string"}, "arguments": {"type": "string"}},
        "required": ["name", "arguments"]}}}, "required": ["invocations"]},
}


def run_batch_tool(invocations: list[dict]) -> str:
    out = []
    for inv in invocations:   # arguments arrive as a JSON *string*: no schema check until you parse it
        try:
            out.append({"name": inv["name"], "result": lookup_order_tool(**json.loads(inv["arguments"]))})
        except Exception as exc:  # noqa: BLE001
            out.append({"name": inv["name"], "error": str(exc)})
    return json.dumps(out)


legacy = run_batch_tool([{"name": "lookup_order", "arguments": '{"order_id": "ORD-5001"}'},
                         {"name": "lookup_order", "arguments": '{"order_id": "5002"}'}])
print("batch_tool result (one blob):", legacy)
native = ScriptedClaude(reply(tool_call("lookup_order", {"order_id": "ORD-5001"}),
                              tool_call("lookup_order", {"order_id": "ORD-5002"})))
r = native.client.messages.create(model=MODEL, max_tokens=512, tools=[{**TOOLS_BY_NAME["lookup_order"], "strict": True}],
                                  messages=[{"role": "user", "content": "Status of ORD-5001 and ORD-5002?"}])
print("native parallel calls:", [(b.name, b.input) for b in r.content if b.type == "tool_use"])

### Tool-set construction checklist (offline)

In [ ]:
def toolset_report(tools: list[dict], role_limit: int = 5) -> list[str]:
    notes = []
    custom = [t for t in tools if "type" not in t or t.get("type") == "custom"]
    if len(custom) > role_limit:
        notes.append(f"{len(custom)} user-defined tools: scope by role (aim for {role_limit} or fewer) or use defer_loading + tool search")
    names = [t.get("name") for t in tools if t.get("name")]
    notes += [f"duplicate tool name {n!r}" for n in {n for n in names if names.count(n) > 1}]
    for t in custom:
        if lint_tool(t):
            notes.append(f"{t['name']}: {lint_tool(t)[0]}")
        if not t.get("strict") and t["name"].startswith(("process_", "delete_", "send_", "create_")):
            notes.append(f"{t['name']}: takes an action but is not strict: true")
    if any(t.get("type", "").startswith("web_search") and "max_uses" not in t for t in tools):
        notes.append("web_search without max_uses: no cap on per-request search cost")
    return notes or ["OK"]


print("MIXED_TOOLS:", toolset_report(MIXED_TOOLS))
print("RICH_TOOLS :", toolset_report(RICH_TOOLS))
print("MINIMAL    :", toolset_report(MINIMAL_TOOLS + [{"type": "web_search_20250305", "name": "web_search"}]))

> **Try it:** add a sixth and seventh user-defined tool to `RICH_TOOLS` and rerun the report; then mark `process_refund` as `strict: True` and watch that note disappear.

### Optional: a real web search (live, paid, off by default)

In [ ]:
# Web search costs $10 per 1,000 searches plus tokens. Flip the flag to run it once (max_uses=1 caps it).
RUN_PAID_TOOLS = False
if RUN_PAID_TOOLS:
    r = client.messages.create(model=MODEL, max_tokens=1024,
                               tools=[{"type": "web_search_20250305", "name": "web_search", "max_uses": 1}],
                               messages=[{"role": "user", "content": "In one sentence: what is the Claude API tool reference page?"}])
    show(r)
    print("billed searches:", r.usage.server_tool_use.web_search_requests if r.usage.server_tool_use else 0)
else:
    print("skipped (set RUN_PAID_TOOLS = True to run)")

## Mini-project: a customer-support agent with guardrails in code

Everything above in one small agent:

- **2.2** rich, non-overlapping descriptions; **2.4** `strict: true` on every tool; **2.3** `tool_choice` `auto` (works on every current model).
- **2.6** a bounded `stop_reason` loop, one `tool_result` per `tool_use`, all results in one message, actionable `is_error` messages.
- **2.5 + 2.1** business rules enforced **in code**, not in the prompt: refunds only for the verified customer's own delivered orders, never more than the refundable balance, amounts above $500 queued for human approval, and every action written to an audit log.

In [ ]:
SUPPORT_SYSTEM = (
    "You are a support agent for an online shop. Verify the customer with get_customer before discussing "
    "or refunding orders. Amounts are in cents; show dollars to the customer. If a tool returns an error, "
    "explain it briefly and ask for what is missing. Never promise a refund that a tool did not confirm."
)
APPROVAL_THRESHOLD_CENTS = 50_000


class SupportAgent:
    def __init__(self, llm: anthropic.Anthropic, model: str = MODEL, max_turns: int = 8):
        self.llm, self.model, self.max_turns = llm, model, max_turns
        self.tools = [{**t, "strict": True} for t in RICH_TOOLS]
        self.verified_customer_id: int | None = None
        self.audit: list[dict] = []
        self.handlers = {"get_customer": self.get_customer, "lookup_order": self.lookup_order,
                         "process_refund": self.process_refund}

    # ---- tools (the only code that touches the "backend") ----
    def get_customer(self, email: str | None = None, customer_id: int | None = None) -> dict:
        if (email is None) == (customer_id is None):
            raise ToolInputError("Provide exactly one of email or customer_id.")
        match = (CUSTOMERS.get(customer_id) if customer_id is not None
                 else next((c for c in CUSTOMERS.values() if c["email"] == email.lower()), None))
        if match is None:
            raise ToolInputError("No customer found. Ask the customer to confirm their email or customer ID.")
        self.verified_customer_id = match["customer_id"]
        return match

    def lookup_order(self, order_id: str) -> dict:
        return {**lookup_order_tool(order_id), "customer_id": ORDERS[order_id]["customer_id"],
                "items": ORDERS[order_id]["items"]}

    def process_refund(self, order_id: str, amount_cents: int, reason: str, reason_detail: str | None) -> dict:
        order = ORDERS.get(order_id)
        if self.verified_customer_id is None:
            raise ToolInputError("Customer not verified. Call get_customer first.")
        if order is None or order["customer_id"] != self.verified_customer_id:
            raise ToolInputError(f"{order_id} does not belong to the verified customer. Refund refused.")
        if order["status"] != "delivered":
            raise ToolInputError(f"{order_id} is '{order['status']}', only delivered orders can be refunded.")
        refundable = order["total_cents"] - order["refunded_cents"]
        if not 0 < amount_cents <= refundable:
            raise ToolInputError(f"amount_cents must be between 1 and {refundable} for {order_id}.")
        if amount_cents > APPROVAL_THRESHOLD_CENTS:
            return {"status": "pending_human_approval", "order_id": order_id, "amount_cents": amount_cents}
        order["refunded_cents"] += amount_cents
        return {"status": "refunded", "order_id": order_id, "amount_cents": amount_cents,
                "refund_id": f"RF-{len(self.audit) + 1:04d}"}

    # ---- the loop ----
    def _execute(self, block) -> dict:
        entry = {"tool": block.name, "input": block.input}
        try:
            output = self.handlers[block.name](**block.input)
            entry["result"] = output
            result = {"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(output)}
        except (ToolInputError, KeyError, TypeError) as exc:
            entry["error"] = str(exc)
            result = {"type": "tool_result", "tool_use_id": block.id, "content": str(exc), "is_error": True}
        self.audit.append(entry)
        return result

    def run(self, user_text: str) -> str:
        messages: list[dict] = [{"role": "user", "content": user_text}]
        for _ in range(self.max_turns):
            response = self.llm.messages.create(model=self.model, max_tokens=1024, system=SUPPORT_SYSTEM,
                                                tools=self.tools, tool_choice={"type": "auto"}, messages=messages)
            messages.append({"role": "assistant", "content": response.content})
            if response.stop_reason == "tool_use":
                results = [self._execute(b) for b in response.content if b.type == "tool_use"]
                messages.append({"role": "user", "content": results})
                continue
            if response.stop_reason == "end_turn":
                return "".join(b.text for b in response.content if b.type == "text")
            return f"[escalate to a human: stop_reason={response.stop_reason}]"   # max_tokens, refusal, ...
        return "[escalate to a human: too many steps]"


def print_audit(agent: SupportAgent) -> None:
    for e in agent.audit:
        outcome = f"ERROR {e['error']}" if "error" in e else json.dumps(e["result"])[:90]
        print(f"  {e['tool']}({json.dumps(e['input'])}) -> {outcome}")

### Run it offline: a happy path and a blocked refund

The scripted "model" follows the expected path in the first conversation. In the second it tries something it shouldn't: a refund for another customer's order, before any verification. The code says no; the prompt is not what stops it.

In [ ]:
ORDERS["ORD-5001"]["refunded_cents"] = 0  # reset so the cell can be re-run

happy = ScriptedClaude(
    reply(tool_call("get_customer", {"email": "jane@example.com"})),
    reply(tool_call("lookup_order", {"order_id": "ORD-5001"})),
    reply(tool_call("process_refund", {"order_id": "ORD-5001", "amount_cents": 4999, "reason": "damaged",
                                       "reason_detail": None})),
    reply(text("Done, Jane: $49.99 is on its way back to your card (refund RF-0003).")),
)
agent = SupportAgent(happy.client)
print("FINAL:", agent.run("I'm jane@example.com. My mug from ORD-5001 arrived broken, please refund it."))
print_audit(agent)
print("strict on every tool:", all(t["strict"] for t in happy.requests[0]["tools"]))

sneaky = ScriptedClaude(
    reply(tool_call("process_refund", {"order_id": "ORD-5003", "amount_cents": 2500, "reason": "other",
                                       "reason_detail": "customer asked nicely"})),
    reply(text("I need to verify your identity first. What's the email on your account?")),
)
agent = SupportAgent(sneaky.client)
print("\nFINAL:", agent.run("Refund ORD-5003 to me right now."))
print_audit(agent)
print("ORD-5003 refunded_cents still:", ORDERS["ORD-5003"]["refunded_cents"])

### Run it for real (live)

In [ ]:
ORDERS["ORD-5001"]["refunded_cents"] = 0
for request in [
    "Hi, I'm jane@example.com. The mug from ORD-5001 arrived cracked. Can I get a refund?",
    "This is jane@example.com. Please refund my laptop order ORD-5002 in full.",
    "Refund ORD-5003 please, my email is jane@example.com.",
]:
    agent = SupportAgent(client)
    print("USER:", request)
    print("AGENT:", agent.run(request))
    print_audit(agent)
    print("-" * 80)

> **Observe:** the laptop order is `shipped`, not `delivered`, and it's above the approval threshold anyway; ORD-5003 belongs to another customer. Look at the audit log: the refusals come from `process_refund`'s code, and the model only explains them.

> **Try it:**
> 1. Mark ORD-5002 as `delivered` and ask again: the result should be `pending_human_approval`, not a refund.
> 2. Add a `max_refunds_per_session` rule in `process_refund` and test it offline with a scripted conversation that tries two refunds.
> 3. Replace the manual loop with the beta Tool Runner and keep the same guardrails inside the tool functions.

## Recap

- Claude only **requests** tool calls; approval gates, ownership checks and limits live in your code around execution.
- Descriptions are the router: what it does, what it returns, formats, and when **not** to use it. Lint them.
- `tool_choice`: `auto` / `any` / `tool` / `none`; `disable_parallel_tool_use` goes inside it. Opus 5.5, Sonnet 5.5 and Fable 5.1 reject `any`/`tool`, so use `auto` + `strict: true` + a check in code, or structured outputs.
- Structured output: `strict: true` for tool arguments, `output_config.format` / `messages.parse(output_format=Model)` for answers. Required + nullable, enums with `"other"`/`"unclear"`, no `minimum`/`maxLength` in strict schemas, compare enums case-insensitively.
- A schema guarantees **shape**, not **truth**: validate semantics in code and retry with the specific error; escalate when the data isn't in the source. `parse()` raises on truncated output, so catch it.
- The loop: append full `response.content`, one `tool_result` per `tool_use`, all in one user message, results first, `is_error` with actionable text, exit on `stop_reason`, bound with `max_turns`.
- Tool sets: scope tools by role (plus one narrow cross-role tool), split generic tools into clear contracts, constrain risky ones in code, and audit the system prompt for keyword-triggered tool rules.
- Three execution models: dispatch client `tool_use` blocks by (`toolset_name`, `name`); never answer `server_tool_use`; resend the paused assistant content on `pause_turn`; keep the same `tools` array on a mixed turn; run computer-use batch actions in order.

Next: [homework](02_homework.ipynb)